In [ ]:
# Copyright 2026 The Google Research Authors.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Generating Canaries for Privacy Auditing

This notebook generates two types of canary datasets for use with the
`privacy_auditing` codebase:

1. **Synthetic temperature canaries** — emails sampled from a Gemma 3 12B IT
   model at various temperatures. Higher temperatures produce more diverse
   (and harder-to-memorize) canaries.
2. **PERP canaries** — canaries derived from real Enron test emails via an
   adaptive temperature-search procedure that targets a specific perplexity
   range, following the construction from Meeus et al.

## Prerequisites

- A Gemma 3 12B IT checkpoint (set `GEMMA3_12B_IT_CKPT_PATH` below)
- A Gemma 3 tokenizer (set `GEMMA3_TOKENIZER_PATH` below)
- The Enron email dataset in `.bagz` format (set `ENRON_TRAIN_PATH` and
  `ENRON_TEST_PATH` below)
- An output directory for writing canary files (set `OUTPUT_DIR` below)

Install dependencies:
```
pip install -r requirements.txt
pip install array-record  # for ArrayRecord I/O
```

## Configuration

Set these paths before running the notebook.

In [ ]:
import os

# -----------------------------------------------------------------------
# UPDATE THESE PATHS BEFORE RUNNING
# -----------------------------------------------------------------------
# Path to the Gemma 3 12B IT model checkpoint directory.
GEMMA3_12B_IT_CKPT_PATH = os.environ.get(
    'GEMMA3_12B_IT_CKPT_PATH', '<PATH_TO_GEMMA3_12B_IT_CHECKPOINT>'
)
# Path to the Gemma 3 tokenizer model file.
GEMMA3_TOKENIZER_PATH = os.environ.get(
    'GEMMA3_TOKENIZER_PATH', '<PATH_TO_GEMMA3_TOKENIZER>'
)
# Path to the Enron training set in .bagz format.
ENRON_TRAIN_PATH = os.environ.get(
    'ENRON_TRAIN_PATH', '<PATH_TO_ENRON_TRAIN_BAGZ>'
)
# Path to the Enron test set in .bagz format.
ENRON_TEST_PATH = os.environ.get(
    'ENRON_TEST_PATH', '<PATH_TO_ENRON_TEST_BAGZ>'
)
# Directory to write generated canary files.
OUTPUT_DIR = os.environ.get('OUTPUT_DIR', '/tmp/privacy_auditing_canaries')
# -----------------------------------------------------------------------

## Imports and Setup

In [ ]:
from IPython import display
from tunix.generate.tokenizer_adapter import TokenizerAdapter, Tokenizer
from tunix.generate import sampler as sampler_lib
from tunix.models.gemma3 import model as model_lib
from grain import python as pygrain

import glob
import json
import math
import matplotlib.pyplot as plt
import numpy as np
import os
from tqdm.auto import tqdm

# gfile shim: use tensorflow.io.gfile if available, otherwise fall back to stdlib.
try:
  from tensorflow.io import gfile
except ImportError:
  import glob as _glob
  class gfile:
    @staticmethod
    def Glob(pattern): return _glob.glob(pattern)
    @staticmethod
    def Exists(path): return os.path.exists(path)
    @staticmethod
    def MakeDirs(path): os.makedirs(path, exist_ok=True)
    @staticmethod
    def Open(path, mode='r'):
      return open(path, mode)
    @staticmethod
    def SetMode(path, mode): os.chmod(path, mode)

## Load Model and Tokenizer

In [ ]:
tokenizer = Tokenizer(
    tokenizer_path=GEMMA3_TOKENIZER_PATH,
    add_bos=False,
    add_eos=False,
)

In [ ]:
# Load Gemma 3 12B IT model from checkpoint.
# NOTE: Replace params_lib.create_model_from_checkpoint with your model-loading
# implementation. The tunix OSS library provides load_and_shard_model() or
# equivalent — see https://github.com/google-deepmind/tunix for details.
import tunix

model_config = model_lib.ModelConfig.gemma3_12b_it()
ckpt_path = GEMMA3_12B_IT_CKPT_PATH

# TODO: Replace with your model loading implementation.
# Example (tunix OSS):
#   gemma = tunix.load_and_shard_model(ckpt_path, model_config)
raise NotImplementedError(
    f'Implement model loading from checkpoint: {ckpt_path}\\n'
    'See https://github.com/google-deepmind/tunix for the OSS API.'
)

In [ ]:
sampler = sampler_lib.Sampler(
    transformer=gemma,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=768,
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    ),
)

In [ ]:
# Quick sanity check — generate one sample from the model.
eot_id = tokenizer.encode('<end_of_turn>')[0]

sampler(
    input_strings=['<start_of_turn>user\nTell me something very cool!<end_of_turn>\n<start_of_turn>model\n'],
    max_generation_steps=200,
    temperature=1,
    forbidden_tokens=[tokenizer.eos_id()],
    top_p=1.0,
    seed=30,
    eos_tokens=[eot_id],
).text

# Synthetic Temperature Canaries

Generate synthetic email canaries by prompting Gemma 3 12B IT at various
sampling temperatures. Each temperature produces a distinct distribution of
canary text: lower temperatures give more conservative, repetitive outputs while
higher temperatures give more diverse outputs.

6000 canaries are generated per temperature setting: 3000 "seen" (used in
training) and 3000 "unseen" (held out for auditing). Canaries that fail an
idempotency check (encode → decode → encode round-trip) or are shorter than
`canary_len` tokens are discarded.

Outputs are written as sharded ArrayRecord files to `OUTPUT_DIR`.

In [ ]:
import tensorflow as tf
import os
from typing import Iterable
import math
from array_record.python import array_record_module
import random
import numpy as np
import itertools


def write_array(path: str, data: np.ndarray) -> None:
  """Writes a numpy array to a file using np.save."""
  outfile_dir = os.path.dirname(path)
  if outfile_dir and not gfile.Exists(outfile_dir):
    gfile.MakeDirs(outfile_dir)
  with gfile.Open(path, 'wb') as f:
    np.save(f, data)


def write_examples(
    path: str,
    examples: Iterable[tf.train.Example],
    shard_count: int,
    total_examples: int,
) -> None:
  """Writes a set of tf.train.Example protos as sharded ArrayRecord files."""
  outfile_dir = os.path.dirname(path)
  if outfile_dir and not gfile.Exists(outfile_dir):
    gfile.MakeDirs(outfile_dir)

  examples_per_shard = math.ceil(total_examples / shard_count)
  digits = len(str(shard_count))
  examples_iter = iter(examples)
  examples_written = 0
  for shard in range(shard_count):
    outfile = f'{path}.arrayrecord-{shard:0{digits}}-of-{shard_count:0{digits}}'
    end_shard = min(examples_written + examples_per_shard, total_examples)
    writer = array_record_module.ArrayRecordWriter(outfile, 'group_size:1')
    try:
      while examples_written < end_shard:
        writer.write(next(examples_iter).SerializeToString())
        examples_written += 1
    finally:
      writer.close()


def make_example(canary: str) -> tf.train.Example:
  """Wraps a canary string as a tf.train.Example proto."""
  def _bytes_feature(value: bytes) -> tf.train.Feature:
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))
  return tf.train.Example(
      features=tf.train.Features(
          feature={
              'BODY_NORMALIZED_LIST': _bytes_feature(canary.encode('utf-8'))
          }
      )
  )


# Formatting constants — canaries are framed with a Subject/Body prefix and an
# end-of-turn suffix so they match the IT-formatted training distribution.
subject = '0'
prefix = 'Subject:\n' + subject + '\nBody:\n'
suffix = '<end_of_turn>'

prefix_len = len(tokenizer.encode(prefix))
suffix_len = len(tokenizer.encode(suffix))


def extract_canary_tokens(full_canary_tokens):
  return full_canary_tokens[prefix_len:-suffix_len]


def check_canary_idempotent(canary: str) -> bool:
  """Returns True iff encode→decode→encode is lossless for this canary."""
  if not canary:
    return False
  canary_tokens = tokenizer.encode(canary)
  full_canary = prefix + canary + suffix
  full_canary_tokens = tokenizer.encode(full_canary)
  decoded = tokenizer.decode(full_canary_tokens)
  return (
      decoded == full_canary
      and extract_canary_tokens(full_canary_tokens) == canary_tokens
  )

In [ ]:
eot_id = tokenizer.encode('<end_of_turn>')[0]
canary_len = 60  # Minimum number of tokens required for a valid canary.

# Prompt used to generate synthetic email canaries.
prompts = ["""\
<start_of_turn>user
Generate a realistic email of about 100 words that could plausibly have been an internal \
communication of a large American energy and commodities company during the \
years 1998-2002. There are no smartphones, Slack, or Zoom. Communications are \
via desktop Outlook/Lotus Notes. Focus on the "Merchant Energy" sector. Possible \
topics include the trading floor, internal politics, logistics, or banal corporate life. \
Add some interesting details so the email is likely to be unique. \
Do not include any meta-commentary. Respond only with the email body, nothing else. Please avoid placeholders for names and places, etc.: use only realistic names.
<end_of_turn>
<start_of_turn>model
"""]


def generate_canaries(temperature: float, canaries_per_set: int = 3000):
  """Generates synthetic email canaries at the given sampling temperature.

  Samples batches of 64 emails from the model using the prompt above until
  ``2 * canaries_per_set`` idempotent, long-enough canaries are collected.
  The canaries are split into seen/unseen sets of ``canaries_per_set`` each
  and written to ``OUTPUT_DIR`` as sharded ArrayRecord files.

  Args:
    temperature: Sampling temperature.  Try values in [0.8, 1.0, 1.5, 2.0,
      2.5, 3.0].
    canaries_per_set: Number of canaries in each of the seen and unseen sets
      (default 3000).

  Returns:
    A (seen, unseen) tuple of lists of tf.train.Example protos.
  """
  num_canaries = 2 * canaries_per_set
  valid_canaries = []
  non_idempotent = 0
  short = 0
  pbar = tqdm(total=num_canaries)

  for i in itertools.count():
    if len(valid_canaries) == num_canaries:
      break
    canaries = sampler(
        input_strings=prompts * 64,
        max_generation_steps=200,
        temperature=temperature,
        top_p=1.0,
        seed=i + int(1000000 * temperature),
        eos_tokens=[eot_id],
    )
    for canary_tokens in canaries.tokens:
      if len(canary_tokens) < canary_len:
        short += 1
        continue
      canary_text = tokenizer.decode(canary_tokens.tolist())
      if check_canary_idempotent(canary_text):
        valid_canaries.append(canary_text)
        pbar.update(1)
        if len(valid_canaries) == num_canaries:
          break
      else:
        non_idempotent += 1

  pbar.close()

  examples = [make_example(c) for c in valid_canaries]
  seen = examples[:canaries_per_set]
  unseen = examples[canaries_per_set:]

  temperature_str = str(temperature).replace('.', '_')
  seen_path = os.path.join(
      OUTPUT_DIR, f'synthetic_enron_temp{temperature_str}.seen'
  )
  unseen_path = os.path.join(
      OUTPUT_DIR, f'synthetic_enron_temp{temperature_str}.unseen'
  )

  write_examples(seen_path, seen, 10, len(seen))
  write_examples(unseen_path, unseen, 10, len(unseen))

  print(f'Wrote {len(seen)} seen  → {seen_path}')
  print(f'Wrote {len(unseen)} unseen → {unseen_path}')
  print(f'Non-idempotent discarded: {non_idempotent}')
  print(f'Too short discarded:      {short}')

  return seen, unseen

In [ ]:
# Generate canaries at each temperature. Comment out temperatures you don't need.
# temp0p8_seen, temp0p8_unseen = generate_canaries(0.8)
# temp1_seen,   temp1_unseen   = generate_canaries(1.0)
# temp1p5_seen, temp1p5_unseen = generate_canaries(1.5)
# temp2_seen,   temp2_unseen   = generate_canaries(2.0)
temp2p5_seen, temp2p5_unseen = generate_canaries(2.5)
temp3_seen,   temp3_unseen   = generate_canaries(3.0)

In [ ]:
def inspect_canaries(temperature: float, n: int = 3):
  """Reads back generated canaries from disk and prints random examples."""
  temperature_str = str(temperature).replace('.', '_')
  pattern = os.path.join(
      OUTPUT_DIR,
      f'synthetic_enron_temp{temperature_str}.seen.arrayrecord*',
  )
  canary_paths = gfile.Glob(pattern)
  if not canary_paths:
    print(f'No files found for temperature {temperature} at {pattern}')
    return []

  canaries = pygrain.ArrayRecordDataSource(canary_paths)
  canonical_ex = []
  for i, ex in enumerate(iter(canaries)):
    if i > 250:
      break
    ex = pygrain.fast_proto.parse_tf_example_experimental(ex)
    arr = ex['BODY_NORMALIZED_LIST']
    text = arr.item()
    canonical_ex.append(text.decode('utf8'))
  return canonical_ex


rng = np.random.default_rng()
for temp in [0.8, 1.0, 1.5, 2.0, 2.5, 3.0]:
  examples = inspect_canaries(temp)
  if not examples:
    continue
  print(f'=== Temperature: {temp} ===')
  sample = rng.choice(examples, size=min(3, len(examples)), replace=False)
  for example in sample:
    print(example)
    print('----------')
  print()

# PERP Canaries

PERP canaries are derived from real Enron *test* emails using an adaptive
temperature-search procedure.  Starting from a real email prefix, the model
generates a suffix at a sampled temperature.  The temperature is adjusted
iteratively (per-example) to hit a target perplexity window
`[min_perp, max_perp]`.  Canaries that fall outside this window are retried
at a shifted temperature.

This construction follows:
> Matthieu Meeus, Lukas Wutschitz, Santiago Zanella-Béguelin, Shruti Tople,
> Reza Shokri (2025).
> *The Canary's Echo: Auditing Privacy Risks of LLM-Generated Synthetic Text*

## Step 1: Load the Enron dataset

In [ ]:
import bagz


def read_from_bagz(file: str) -> list[str]:
  """Reads all text records from a .bagz file."""
  canaries = []
  with bagz.BagReader(file) as reader:
    num_records = len(reader)
    print(f'Reading {num_records} records from {file}')
    for record in reader:
      text = pygrain.fast_proto.parse_tf_example_experimental(record)[
          'body'
      ].item().decode('utf8')
      canaries.append(text)
  return canaries


def get_canaries_from_arrayrecord(pattern: str) -> list[str]:
  """Reads canary strings from sharded ArrayRecord files matching pattern."""
  def extract_text(ex):
    return (
        pygrain.fast_proto.parse_tf_example_experimental(ex)[
            'BODY_NORMALIZED_LIST'
        ]
        .item()
        .decode('utf8')
    )

  canaries = []
  for path in gfile.Glob(pattern):
    reader = array_record_module.ArrayRecordReader(path)
    for record in reader.read_all():
      canaries.append(extract_text(record))
  return canaries


enron_train = read_from_bagz(ENRON_TRAIN_PATH)
enron_test = read_from_bagz(ENRON_TEST_PATH)

## Step 2: Filter test examples that are not near any training example

To ensure canaries are not already memorized from training, we filter the test
set to keep only examples whose 30-token prefix:

1. Does **not** appear verbatim in the training set (hash filter), and
2. Is **not** within edit distance 5 of any training prefix (edit-distance filter).

In [ ]:
import concurrent.futures
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
from tqdm.auto import tqdm

# Build a hash set of all 30-token windows in the training corpus.
rng = np.random.default_rng(42)
HASH_WEIGHTS = rng.integers(1, 2**64 - 1, size=30, dtype=np.uint64)

enron_train_tokens = [
    tokenizer.encode(text)
    for text in tqdm(enron_train, desc='Tokenizing train texts')
]


def get_hashes_from_tokens(tokens):
  if len(tokens) < 30:
    return set()
  arr = np.array(tokens, dtype=np.uint64)
  windows = sliding_window_view(arr, 30)
  hashes = windows.dot(HASH_WEIGHTS)
  return set(hashes.tolist())


phrase_hashes = set()
with concurrent.futures.ThreadPoolExecutor() as executor:
  for hashes in tqdm(
      executor.map(get_hashes_from_tokens, enron_train_tokens),
      total=len(enron_train_tokens),
      desc='Hashing training phrases',
  ):
    phrase_hashes.update(hashes)

print(f'Total unique 30-token phrases (hashes): {len(phrase_hashes)}')

In [ ]:
train_prefixes_set = {
    tuple(t[:30]) for t in enron_train_tokens if len(t) >= 30
}
train_prefixes = np.array(list(train_prefixes_set), dtype=np.int32)

test_data = []
for text in tqdm(enron_test, desc='Tokenizing test examples'):
  tokens = tokenizer.encode(text)
  if len(tokens) >= 30:
    test_data.append((text, tokens[:30]))

In [ ]:
import numba


@numba.njit(nogil=True)
def check_edit_distances(test_prefix, train_prefixes, threshold=10):
  """Returns True iff test_prefix is >threshold edit distance from all training prefixes."""
  m = 30
  for i in range(train_prefixes.shape[0]):
    train_p = train_prefixes[i]
    dp = np.arange(m + 1, dtype=np.int32)
    for j in range(m):
      new_dp = np.empty(m + 1, dtype=np.int32)
      new_dp[0] = j + 1
      min_val = new_dp[0]
      for k in range(m):
        cost = 0 if test_prefix[j] == train_p[k] else 1
        new_dp[k + 1] = min(dp[k + 1] + 1, new_dp[k] + 1, dp[k] + cost)
        if new_dp[k + 1] < min_val:
          min_val = new_dp[k + 1]
      dp = new_dp
      # Early exit if minimum possible distance already exceeds threshold.
      if min_val > threshold:
        break
    if dp[m] <= threshold:
      return False  # Too close to a training example.
  return True  # Sufficiently far from all training examples.


def is_valid_test_example(item):
  text, prefix = item
  prefix_arr = np.array(prefix, dtype=np.uint64)

  # Property 1: prefix not in training set (exact hash match).
  if prefix_arr.dot(HASH_WEIGHTS) in phrase_hashes:
    return None

  # Property 2: prefix not within edit distance 5 of any training prefix.
  if not check_edit_distances(
      np.array(prefix, dtype=np.int32), train_prefixes, 5
  ):
    return None

  return text


valid_test_examples = []
with concurrent.futures.ThreadPoolExecutor() as executor:
  results = list(
      tqdm(
          executor.map(is_valid_test_example, test_data),
          total=len(test_data),
          desc='Filtering test examples',
      )
  )

valid_test_examples = set(res for res in results if res is not None)
print(f'Found {len(valid_test_examples)} valid test examples.')

In [ ]:
# Optionally, save valid_test_examples to disk for reuse across sessions.
valid_examples_path = os.path.join(OUTPUT_DIR, 'valid_test_examples.bagz')

if not gfile.Exists(OUTPUT_DIR):
  gfile.MakeDirs(OUTPUT_DIR)

with bagz.BagWriter(valid_examples_path) as writer:
  for text in valid_test_examples:
    writer.write(text.encode('utf-8'))

print(f'Saved {len(valid_test_examples)} valid test examples → {valid_examples_path}')

## Step 3: Create a smaller-cache sampler for PERP generation

PERP generation runs many short (30-token) suffix samples, so a smaller KV
cache is used here to reduce memory pressure.

In [ ]:
sampler = sampler_lib.Sampler(
    transformer=gemma,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=96,
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    ),
)


def sample(texts, prefix_len, suffix_len, temp, seed):
  """Samples suffixes of length suffix_len from model given text prefixes."""
  tokens = [tokenizer.encode(text) for text in texts]
  prefixes = [t[0:prefix_len] for t in tokens]
  prefix_texts = [tokenizer.decode(prefix) for prefix in prefixes]
  suffixes = sampler(
      input_strings=prefix_texts,
      max_generation_steps=suffix_len,
      temperature=temp,
      top_p=1.0,
      seed=seed,
  ).tokens
  padded_suffixes = [list(s) + [0] * (suffix_len - len(s)) for s in suffixes]
  return np.array(prefixes), np.array(padded_suffixes)

In [ ]:
import jax
import jax.numpy as jnp


def make_causal_attn_mask(input_mask: jax.Array) -> jax.Array:
  """Creates a causal attention mask for a batch.

  Args:
    input_mask: Boolean array of shape (batch_size, seq_len).

  Returns:
    Boolean array of shape (batch_size, seq_len, seq_len).
  """
  seq_len = input_mask.shape[-1]
  attn_mask = input_mask[..., None, :]
  causal_mask = jnp.tril(jnp.ones((seq_len, seq_len), dtype=jnp.bool_))
  return attn_mask * causal_mask[None, ...]


def compute_perplexity_batch(
    model: model_lib.Gemma3,
    prefix: jax.Array | np.ndarray,
    suffix: jax.Array | np.ndarray,
) -> jax.Array:
  """Computes per-example perplexity of suffixes given prefixes.

  Args:
    model: The language model.
    prefix: Integer array of shape (batch_size, prefix_len).
    suffix: Integer array of shape (batch_size, suffix_len).

  Returns:
    Float array of shape (batch_size,) with perplexity for each example.
  """
  if prefix.ndim != 2 or suffix.ndim != 2:
    raise ValueError('prefix and suffix must be 2D arrays for batch processing.')
  if prefix.shape[0] != suffix.shape[0]:
    raise ValueError('Batch sizes of prefix and suffix must match.')

  batch_size = prefix.shape[0]
  prefix_len = prefix.shape[1]
  suffix_len = suffix.shape[1]
  seq_len = prefix_len + suffix_len

  if suffix_len == 0 or seq_len == 0:
    return jnp.ones(batch_size, dtype=jnp.float32)

  full_sequence = jnp.concatenate([prefix, suffix], axis=-1)
  positions = jnp.arange(seq_len)[None, :]
  input_mask = jnp.ones((batch_size, seq_len), dtype=jnp.bool_)
  attention_mask = make_causal_attn_mask(input_mask)

  model.eval()
  next_token_logits, _ = model(
      last_tokens=full_sequence,
      positions=positions,
      cache=None,
      attention_mask=attention_mask,
  )

  next_token_logits = next_token_logits.astype(jnp.float32)
  log_probs = jax.nn.log_softmax(next_token_logits, axis=-1)

  targets = full_sequence[:, 1:]
  logits_for_targets = log_probs[:, :-1, :]
  log_probs_of_sequence = jnp.take_along_axis(
      logits_for_targets, targets[..., None], axis=-1
  ).squeeze(-1)

  start_index = max(0, prefix_len - 1)
  end_index = seq_len - 1
  time_indices = jnp.arange(seq_len - 1)[None, :]
  mask = (time_indices >= start_index) & (time_indices < end_index)
  valid_mask = targets != 0
  combined_mask = mask & valid_mask

  selected_log_probs = log_probs_of_sequence * combined_mask
  sum_log_probs = jnp.sum(selected_log_probs, axis=-1)
  actual_suffix_lens = jnp.maximum(jnp.sum(combined_mask, axis=-1), 1.0)

  return jnp.exp(-sum_log_probs / actual_suffix_lens)

## Step 4: Generate PERP canaries

In [ ]:
import itertools
import time
from collections import defaultdict
import numpy as np


def generate_perp(
    enron: list[str],
    initial_temp: float,
    min_perp: float,
    max_perp: float,
    batch_size: int,
    required: int,
) -> list[str]:
  """Generates PERP canaries from Enron test emails.

  Uses an adaptive per-example temperature-search algorithm to produce
  canaries whose model perplexity falls in [min_perp, max_perp].

  Args:
    enron: List of Enron test email strings to use as seed texts.
    initial_temp: Starting sampling temperature.
    min_perp: Minimum acceptable suffix perplexity.
    max_perp: Maximum acceptable suffix perplexity (soft — accepts up to
      2 * max_perp initially then tightens via bucket shifts).
    batch_size: Number of examples to process in each model call.
    required: Total number of PERP canaries to generate.

  Returns:
    A list of ``required`` canary strings.
  """
  num_dup = len(enron) - len(set(enron))
  if num_dup > 0:
    print(f'Warning: {num_dup} duplicate examples in enron')

  buckets = defaultdict(list)
  buckets[0].extend(set(enron))

  perp = []
  non_idempotent_count = wrong_perp_count = 0
  start_time = time.time()

  for i in itertools.count():
    if not buckets or len(perp) >= required:
      break

    iter_start_time = time.time()
    print(
        f'Buckets: {", ".join(f"{t}: {len(b)}" for t, b in sorted(buckets.items()))}'
    )

    temp_bucket, pool = max(
        buckets.items(), key=lambda x: (x[0] > -20, len(x[1]))
    )
    temp = initial_temp * np.exp(temp_bucket * 0.2)

    batch = pool[:batch_size]
    del pool[:batch_size]
    if not pool:
      del buckets[temp_bucket]

    padded_batch = list(itertools.islice(itertools.cycle(batch), batch_size))
    prefixes, suffixes = sample(padded_batch, 30, 30, temp, i)

    if temp_bucket <= -20:
      perps = [(min_perp + max_perp) / 2] * batch_size
    else:
      perps = compute_perplexity_batch(gemma, prefixes, suffixes)

    for ex in batch:
      indices = [idx for idx, e in enumerate(padded_batch) if e == ex]
      success = False

      for idx in indices:
        example_perp = perps[idx]
        if min_perp <= example_perp <= 2 * max_perp:
          m_tokens = prefixes[idx].tolist() + suffixes[idx].tolist()
          m_ex = tokenizer.decode(m_tokens)
          if tokenizer.encode(m_ex) == m_tokens:
            perp.append(m_ex)
            success = True
            break
          non_idempotent_count += 1
        else:
          wrong_perp_count += 1

      if not success and temp_bucket > -20:
        perps_less = np.mean([perps[idx] < min_perp for idx in indices])
        buckets[temp_bucket + int(np.sign(perps_less - 0.5))].append(ex)

    iter_time = time.time() - iter_start_time
    ex_per_sec = ((i + 1) * batch_size) / (time.time() - start_time)
    print(
        f'{i}: {len(perp)} found, {non_idempotent_count} non-idempotent, '
        f'{wrong_perp_count} wrong-perp, {iter_time:.2f}s iter, '
        f'{ex_per_sec:.2f} ex/s'
    )
    print(f'Avg/std perp: {np.mean(perps):.2f}  {np.std(perps):.2f}\n')

  return perp[:required]

In [ ]:
import gc

# Free any unused device memory before the PERP generation loop.
gc.collect()

perp = generate_perp(
    valid_test_examples,
    initial_temp=0.5,
    min_perp=28,
    max_perp=110,
    batch_size=256,
    required=6000,
)

In [ ]:
# Write PERP canaries: first 3000 → seen, remaining 3000 → unseen.
perp_examples = [make_example(canary) for canary in perp]

perp_seen_path = os.path.join(OUTPUT_DIR, 'perp_enron.seen')
perp_unseen_path = os.path.join(OUTPUT_DIR, 'perp_enron.unseen')

write_examples(perp_seen_path, perp_examples[:3000], 1, 3000)
write_examples(perp_unseen_path, perp_examples[3000:], 1, 3000)

print(f'Wrote 3000 seen   → {perp_seen_path}')
print(f'Wrote 3000 unseen → {perp_unseen_path}')